# Bradford Bulls — Frame Reconstruction Pipeline

### Purpose
Takes frames extracted by `02_team_aware_extraction.ipynb` and reconstructs
them into **sharp, clear frames** using video restoration models.

### Pipeline
```
Phase 1: Load metadata CSV from extraction pipeline
Phase 2: For each selected frame:
           → Extract ±8 neighboring frames from video (16-frame window)
           → Scene change detection (trim window at camera cuts)
           → RVRT video deblurring (16 frames → 1 sharp frame)
           → Fallback: NAFNet single-frame deblur (if window too short)
Phase 3: Save reconstructed frames + comparison previews
```

### Requirements
- **GPU**: T4 (Colab free) or better
- **Input**: Frames metadata CSV from `02_team_aware_extraction.ipynb`
- **Output**: Reconstructed sharp frames + updated metadata

### Alternative
This notebook is equivalent to `run_reconstruction.py` (CLI script).
Use the script for batch/automated runs, use this notebook for interactive exploration.

## 0. Install Dependencies
Run this cell, then **Restart Runtime**.

In [ ]:
!pip install -q torch torchvision
!pip install -q timm einops opencv-python-headless

# Clone RVRT repository
import os
if not os.path.exists('/content/RVRT'):
    !git clone https://github.com/JingyunLiang/RVRT.git /content/RVRT
    print('RVRT cloned.')
else:
    print('RVRT already cloned.')

# Download RVRT deblurring weights (GoPro, 16 frames)
RVRT_WEIGHTS_DIR = '/content/RVRT/model_zoo/rvrt'
RVRT_WEIGHTS_FILE = f'{RVRT_WEIGHTS_DIR}/005_RVRT_videodeblurring_GoPro_16frames.pth'
os.makedirs(RVRT_WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(RVRT_WEIGHTS_FILE):
    !wget -q -O {RVRT_WEIGHTS_FILE} \
        'https://github.com/JingyunLiang/RVRT/releases/download/v0.0/005_RVRT_videodeblurring_GoPro_16frames.pth'
    print(f'RVRT weights downloaded: {os.path.getsize(RVRT_WEIGHTS_FILE)/1e6:.0f} MB')
else:
    print('RVRT weights already downloaded.')

# Download NAFNet weights (fallback for short windows)
NAFNET_WEIGHTS_DIR = '/content/nafnet_weights'
NAFNET_WEIGHTS_FILE = f'{NAFNET_WEIGHTS_DIR}/NAFNet-GoPro-width64.pth'
os.makedirs(NAFNET_WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(NAFNET_WEIGHTS_FILE):
    !pip install -q gdown
    !gdown 1S0PVRbyTakYY9a82kujgZLbMihfNBLfC -O {NAFNET_WEIGHTS_FILE}
    print(f'NAFNet weights downloaded: {os.path.getsize(NAFNET_WEIGHTS_FILE)/1e6:.0f} MB')
else:
    print('NAFNet weights already downloaded.')

print('\nDone. Please RESTART RUNTIME now (Runtime > Restart runtime).')

## 1. YOUR CONFIG

In [ ]:
VIDEO_FILENAME = input('Video filename (e.g. M06_black_1080p.mp4): ').strip()
test_input = input('Test mode? Process only first 10 frames (y/n, default=y): ').strip().lower()
TEST_MODE = test_input != 'n'

# RVRT window size — model expects 16 frames
RVRT_WINDOW = 8  # ±8 frames around center = 17 frames, padded/trimmed to 16
# Minimum frames in window to use RVRT (otherwise fallback to NAFNet)
MIN_WINDOW_FOR_RVRT = 7
# Scene change detection threshold
SCENE_CHANGE_THRESHOLD = 0.65

print(f'\n--- Config ---')
print(f'  Video:     {VIDEO_FILENAME}')
print(f'  Mode:      {"TEST (10 frames)" if TEST_MODE else "PRODUCTION (all frames)"}')
print(f'  Window:    ±{RVRT_WINDOW} frames ({RVRT_WINDOW*2+1} total)')
print(f'  Min RVRT:  {MIN_WINDOW_FOR_RVRT} frames')
print(f'  Scene thr: {SCENE_CHANGE_THRESHOLD}')

## 2. Setup Environment

In [ ]:
import torch
import cv2
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('WARNING: No GPU detected. This will be very slow.')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Bradford_Bulls')
VIDEOS_DIR = DRIVE_ROOT / 'videos'
FRAMES_DIR = DRIVE_ROOT / 'frames'
METADATA_DIR = DRIVE_ROOT / 'metadata'
RECONSTRUCTED_DIR = DRIVE_ROOT / 'frames_reconstructed'

MATCH_ID = Path(VIDEO_FILENAME).stem
VIDEO_PATH = VIDEOS_DIR / VIDEO_FILENAME
ORIGINAL_FRAMES_DIR = FRAMES_DIR / MATCH_ID
RECON_OUTPUT_DIR = RECONSTRUCTED_DIR / MATCH_ID
RECON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load metadata from extraction pipeline
CSV_PATH = METADATA_DIR / f'{MATCH_ID}_v3_index.csv'
assert VIDEO_PATH.exists(), f'Video not found: {VIDEO_PATH}'
assert CSV_PATH.exists(), f'Metadata CSV not found: {CSV_PATH}. Run 02_team_aware_extraction.ipynb first.'

df = pd.read_csv(CSV_PATH)
print(f'\nVideo: {VIDEO_PATH.name}')
print(f'Metadata: {len(df)} frames from {CSV_PATH.name}')
print(f'Output: {RECON_OUTPUT_DIR}')

# Video info
cap = cv2.VideoCapture(str(VIDEO_PATH))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Video: {W}x{H} @ {FPS:.0f}fps, {TOTAL_FRAMES:,} frames, {TOTAL_FRAMES/FPS/60:.1f}min')

## 3. Scene Change Detection

Builds a **scene boundary map** for the entire video upfront.
This allows us to quickly trim the frame window for each target frame
without re-scanning the video every time.

In [ ]:
def compute_histogram(frame, bins=64):
    """Compute normalized color histogram for a frame."""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [bins, bins], [0, 180, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()


def detect_scene_boundaries(video_path, frame_nums, window, threshold=0.65):
    """Detect scene changes around target frames.

    Only scans the regions we actually need (around each target frame),
    not the entire video.

    Returns: dict mapping frame_num → (safe_start, safe_end)
    """
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    boundaries = {}

    for center_fn in tqdm(frame_nums, desc='Scene detection'):
        scan_start = max(0, center_fn - window - 2)  # extra buffer
        scan_end = min(total - 1, center_fn + window + 2)

        # Read frames and compute histograms
        cap.set(cv2.CAP_PROP_POS_FRAMES, scan_start)
        prev_hist = None
        scene_cuts = []

        for fn in range(scan_start, scan_end + 1):
            ret, frame = cap.read()
            if not ret:
                break
            hist = compute_histogram(frame)
            if prev_hist is not None:
                diff = cv2.compareHist(
                    prev_hist.reshape(-1, 1).astype(np.float32),
                    hist.reshape(-1, 1).astype(np.float32),
                    cv2.HISTCMP_BHATTACHARYYA
                )
                if diff > threshold:
                    scene_cuts.append(fn)
            prev_hist = hist

        # Find safe window boundaries
        safe_start = max(0, center_fn - window)
        safe_end = min(total - 1, center_fn + window)

        for cut_fn in scene_cuts:
            if cut_fn <= center_fn and cut_fn > safe_start:
                safe_start = cut_fn  # trim start to after the cut
            elif cut_fn > center_fn and cut_fn < safe_end:
                safe_end = cut_fn - 1  # trim end to before the cut

        boundaries[center_fn] = (safe_start, safe_end)

    cap.release()
    return boundaries


# Detect scene boundaries for all target frames
target_frame_nums = df['frame_num'].tolist()
if TEST_MODE:
    target_frame_nums = target_frame_nums[:10]
    print(f'TEST MODE: processing {len(target_frame_nums)} frames')

scene_boundaries = detect_scene_boundaries(
    VIDEO_PATH, target_frame_nums, RVRT_WINDOW, SCENE_CHANGE_THRESHOLD
)

# Report
trimmed = sum(1 for fn, (s, e) in scene_boundaries.items()
              if (fn - s) < RVRT_WINDOW or (e - fn) < RVRT_WINDOW)
short = sum(1 for fn, (s, e) in scene_boundaries.items()
            if (e - s + 1) < MIN_WINDOW_FOR_RVRT)
print(f'\nScene detection results:')
print(f'  Total frames:  {len(scene_boundaries)}')
print(f'  Trimmed by scene cut: {trimmed}')
print(f'  Too short for RVRT (fallback to NAFNet): {short}')

## 4. Load Models

In [ ]:
# ── RVRT Model ────────────────────────────────────────────────────
sys.path.insert(0, '/content/RVRT')
from models.network_rvrt import RVRT as RVRTNet

def load_rvrt(device):
    """Load RVRT deblurring model (GoPro, 16 frames)."""
    model = RVRTNet(
        upscale=1,           # deblurring, not super-resolution
        clip_size=2,
        img_size=[2, 64, 64],
        window_size=[2, 8, 8],
        num_blocks=[1, 2, 1],
        depths=[2, 2, 2],
        embed_dims=[144, 144, 144],
        num_heads=[6, 6, 6],
        inputconv_groups=[1, 1, 1, 1, 1, 1],
        deformable_groups=12,
        attention_heads=12,
        attention_window=[3, 3],
        cpu_cache_length=100,
    )

    weights_path = '/content/RVRT/model_zoo/rvrt/005_RVRT_videodeblurring_GoPro_16frames.pth'
    checkpoint = torch.load(weights_path, map_location='cpu')
    if 'params' in checkpoint:
        state_dict = checkpoint['params']
    elif 'params_ema' in checkpoint:
        state_dict = checkpoint['params_ema']
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict, strict=True)
    model = model.to(device).eval()
    print(f'RVRT loaded on {device}')
    return model


# ── NAFNet Model (fallback) ───────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps
    def forward(self, x):
        mu = x.mean(1, keepdim=True)
        var = (x - mu).pow(2).mean(1, keepdim=True)
        y = (x - mu) / (var + self.eps).sqrt()
        C = self.weight.shape[0]
        return self.weight.reshape(1, C, 1, 1) * y + self.bias.reshape(1, C, 1, 1)

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c, dw_expand=2, ffn_expand=2):
        super().__init__()
        dw_channel = c * dw_expand
        self.conv1 = nn.Conv2d(c, dw_channel, 1)
        self.conv2 = nn.Conv2d(dw_channel, dw_channel, 3, 1, 1, groups=dw_channel)
        self.conv3 = nn.Conv2d(dw_channel // 2, c, 1)
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(dw_channel // 2, dw_channel // 2, 1))
        self.sg = SimpleGate()
        ffn_channel = ffn_expand * c
        self.conv4 = nn.Conv2d(c, ffn_channel, 1)
        self.conv5 = nn.Conv2d(ffn_channel // 2, c, 1)
        self.sg2 = SimpleGate()
        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)))
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)))
    def forward(self, inp):
        x = self.norm1(inp)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg(x)
        x = x * self.sca(x)
        x = self.conv3(x)
        y = inp + x * self.beta
        x = self.conv4(self.norm2(y))
        x = self.sg2(x)
        x = self.conv5(x)
        return y + x * self.gamma

class NAFNet(nn.Module):
    def __init__(self, img_channel=3, width=64, middle_blk_num=1,
                 enc_blk_nums=[1,1,1,28], dec_blk_nums=[1,1,1,1]):
        super().__init__()
        self.intro = nn.Conv2d(img_channel, width, 3, 1, 1)
        self.ending = nn.Conv2d(width, img_channel, 3, 1, 1)
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        chan = width
        for num in enc_blk_nums:
            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
            self.downs.append(nn.Conv2d(chan, 2 * chan, 2, 2))
            chan *= 2
        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])
        for num in dec_blk_nums:
            self.ups.append(nn.Sequential(nn.Conv2d(chan, chan * 2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
        self.padder_size = 2 ** len(enc_blk_nums)
    def forward(self, inp):
        B, C, H, W = inp.shape
        mod_h = (self.padder_size - H % self.padder_size) % self.padder_size
        mod_w = (self.padder_size - W % self.padder_size) % self.padder_size
        inp = F.pad(inp, (0, mod_w, 0, mod_h))
        x = self.intro(inp)
        encs = []
        for encoder, down in zip(self.encoders, self.downs):
            x = encoder(x)
            encs.append(x)
            x = down(x)
        x = self.middle_blks(x)
        for decoder, up, enc_skip in zip(self.decoders, self.ups, encs[::-1]):
            x = up(x)
            x = x + enc_skip
            x = decoder(x)
        x = self.ending(x)
        x = x + inp
        return x[:, :, :H, :W]

def load_nafnet(device):
    """Load NAFNet deblurring model (GoPro, width64)."""
    model = NAFNet()
    weights_path = '/content/nafnet_weights/NAFNet-GoPro-width64.pth'
    checkpoint = torch.load(weights_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get('params', checkpoint)
    model.load_state_dict(state_dict, strict=True)
    model = model.to(device).eval()
    print(f'NAFNet loaded on {device}')
    return model


# Load both models
print('Loading RVRT (video deblurring)...')
rvrt_model = load_rvrt(DEVICE)
print('\nLoading NAFNet (single-frame fallback)...')
nafnet_model = load_nafnet(DEVICE)
print('\nBoth models ready.')

## 5. Reconstruction Functions

In [ ]:
def extract_frame_window(video_path, center_fn, start_fn, end_fn):
    """Extract a window of frames from video."""
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_fn)
    frames = []
    center_idx = None
    for fn in range(start_fn, end_fn + 1):
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        if fn == center_fn:
            center_idx = len(frames) - 1
    cap.release()
    return frames, center_idx


def pad_or_trim_sequence(frames, target_length=16, center_idx=None):
    """Pad or trim frame sequence to exact target_length."""
    n = len(frames)
    if n == target_length:
        return frames, center_idx
    if n > target_length:
        if center_idx is None:
            center_idx = n // 2
        half = target_length // 2
        start = max(0, center_idx - half)
        start = min(start, n - target_length)
        return frames[start:start + target_length], center_idx - start
    padded = list(frames)
    new_center = center_idx if center_idx is not None else n // 2
    while len(padded) < target_length:
        padded.append(padded[-1])
        if len(padded) < target_length:
            padded.insert(0, padded[0])
            new_center += 1
    return padded, new_center


def reconstruct_rvrt(model, frames, center_idx, device,
                     tile_h=256, tile_w=256, overlap_h=20, overlap_w=20):
    """Run RVRT on a sequence of frames, return the reconstructed center frame."""
    imgs = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0 for f in frames]
    tensor = np.stack(imgs, axis=0)
    tensor = torch.from_numpy(tensor).permute(0, 3, 1, 2).unsqueeze(0).to(device)

    _, T, C, H, W = tensor.shape
    pad_h = (8 - H % 8) % 8
    pad_w = (8 - W % 8) % 8
    if pad_h > 0 or pad_w > 0:
        tensor = F.pad(tensor, (0, pad_w, 0, pad_h), mode='reflect')

    _, _, _, pH, pW = tensor.shape

    if pH <= tile_h * 1.5 and pW <= tile_w * 1.5:
        with torch.no_grad(), torch.cuda.amp.autocast():
            output = model(tensor)
        result = output[0, center_idx].cpu().permute(1, 2, 0).numpy()
        result = np.clip(result * 255, 0, 255).astype(np.uint8)[:H, :W]
        return cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

    output_all = torch.zeros(1, T, C, pH, pW, device='cpu')
    weight_map = torch.zeros(1, 1, 1, pH, pW, device='cpu')

    step_h = tile_h - overlap_h
    step_w = tile_w - overlap_w
    y_positions = list(range(0, max(pH - tile_h + 1, 1), step_h))
    if y_positions[-1] + tile_h < pH:
        y_positions.append(pH - tile_h)
    x_positions = list(range(0, max(pW - tile_w + 1, 1), step_w))
    if x_positions[-1] + tile_w < pW:
        x_positions.append(pW - tile_w)

    for y in y_positions:
        for x in x_positions:
            tile = tensor[:, :, :, y:y+tile_h, x:x+tile_w]
            with torch.no_grad(), torch.cuda.amp.autocast():
                out_tile = model(tile).cpu()

            w = torch.ones(1, 1, 1, tile_h, tile_w)
            feather = min(overlap_h // 2, 10)
            if feather > 1:
                for fi in range(feather):
                    a = fi / feather
                    w[:, :, :, fi, :] *= a
                    w[:, :, :, tile_h-1-fi, :] *= a
                    w[:, :, :, :, fi] *= a
                    w[:, :, :, :, tile_w-1-fi] *= a

            output_all[:, :, :, y:y+tile_h, x:x+tile_w] += out_tile * w
            weight_map[:, :, :, y:y+tile_h, x:x+tile_w] += w
            torch.cuda.empty_cache()

    weight_map = torch.clamp(weight_map, min=1e-10)
    output_all = output_all / weight_map

    result = output_all[0, center_idx].permute(1, 2, 0).numpy()
    result = np.clip(result * 255, 0, 255).astype(np.uint8)[:H, :W]
    return cv2.cvtColor(result, cv2.COLOR_RGB2BGR)


def reconstruct_nafnet(model, frame, device, tile_size=256, overlap=32):
    """Single-frame deblurring with NAFNet (fallback)."""
    h, w = frame.shape[:2]
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    result = np.zeros_like(img, dtype=np.float64)
    wmap = np.zeros((h, w, 1), dtype=np.float64)

    step = tile_size - overlap
    y_pos = list(range(0, max(h - tile_size + 1, 1), step))
    if y_pos[-1] + tile_size < h:
        y_pos.append(h - tile_size)
    x_pos = list(range(0, max(w - tile_size + 1, 1), step))
    if x_pos[-1] + tile_size < w:
        x_pos.append(w - tile_size)

    for y in y_pos:
        for x in x_pos:
            tile = img[y:y+tile_size, x:x+tile_size]
            th, tw = tile.shape[:2]
            t = torch.from_numpy(tile).permute(2, 0, 1).unsqueeze(0).to(device)
            with torch.no_grad():
                out = model(t)
            out = out.squeeze(0).permute(1, 2, 0).cpu().numpy()[:th, :tw]

            mask = np.ones((th, tw, 1), dtype=np.float64)
            feather = min(overlap // 2, 16)
            if feather > 1:
                for f in range(feather):
                    a = f / feather
                    mask[f, :] *= a
                    mask[th-1-f, :] *= a
                    mask[:, f] *= a
                    mask[:, tw-1-f] *= a

            result[y:y+th, x:x+tw] += out.astype(np.float64) * mask
            wmap[y:y+th, x:x+tw] += mask

    wmap = np.maximum(wmap, 1e-10)
    result = np.clip(result / wmap * 255, 0, 255).astype(np.uint8)
    return cv2.cvtColor(result, cv2.COLOR_RGB2BGR)


def compute_sharpness(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


print('Reconstruction functions ready.')

## 6. Run Reconstruction

For each frame from the extraction pipeline:
1. Extract neighboring frames (scene-aware window)
2. RVRT reconstruct (or NAFNet fallback)
3. Save reconstructed frame

In [ ]:
RVRT_INPUT_LENGTH = 16
JPEG_QUALITY = 95

results = []
rvrt_count = 0
nafnet_count = 0

for idx, fn in enumerate(tqdm(target_frame_nums, desc='Reconstructing')):
    safe_start, safe_end = scene_boundaries[fn]
    window_size = safe_end - safe_start + 1

    frames_window, center_idx = extract_frame_window(
        VIDEO_PATH, fn, safe_start, safe_end
    )

    if not frames_window or center_idx is None:
        print(f'  SKIP frame {fn}: could not read frames')
        continue

    original = frames_window[center_idx].copy()
    orig_sharpness = compute_sharpness(original)

    try:
        if window_size >= MIN_WINDOW_FOR_RVRT:
            padded, new_center = pad_or_trim_sequence(
                frames_window, RVRT_INPUT_LENGTH, center_idx
            )
            reconstructed = reconstruct_rvrt(
                rvrt_model, padded, new_center, DEVICE
            )
            method = 'rvrt'
            rvrt_count += 1
        else:
            reconstructed = reconstruct_nafnet(
                nafnet_model, original, DEVICE
            )
            method = 'nafnet'
            nafnet_count += 1

        recon_sharpness = compute_sharpness(reconstructed)

        row = df[df['frame_num'] == fn].iloc[0]
        fname = row['filename']
        cv2.imwrite(
            str(RECON_OUTPUT_DIR / fname),
            reconstructed,
            [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
        )

        results.append({
            'frame_num': fn,
            'filename': fname,
            'method': method,
            'window_size': window_size,
            'orig_sharpness': round(orig_sharpness, 1),
            'recon_sharpness': round(recon_sharpness, 1),
            'improvement': round(recon_sharpness / max(orig_sharpness, 1), 2),
        })

        torch.cuda.empty_cache()

    except Exception as e:
        print(f'  ERROR frame {fn}: {e}')
        row = df[df['frame_num'] == fn].iloc[0]
        cv2.imwrite(
            str(RECON_OUTPUT_DIR / row['filename']),
            original,
            [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
        )
        results.append({
            'frame_num': fn,
            'filename': row['filename'],
            'method': 'original',
            'window_size': window_size,
            'orig_sharpness': round(orig_sharpness, 1),
            'recon_sharpness': round(orig_sharpness, 1),
            'improvement': 1.0,
        })

print(f'\n--- Summary ---')
print(f'  RVRT reconstructed: {rvrt_count}')
print(f'  NAFNet fallback:    {nafnet_count}')
print(f'  Total processed:    {len(results)}')

results_df = pd.DataFrame(results)
if len(results_df) > 0:
    avg_improvement = results_df['improvement'].mean()
    print(f'  Avg sharpness improvement: {avg_improvement:.2f}x')

## 7. Save Reconstruction Metadata

In [ ]:
recon_csv_path = METADATA_DIR / f'{MATCH_ID}_reconstruction_index.csv'
results_df.to_csv(recon_csv_path, index=False)
print(f'Reconstruction metadata saved: {recon_csv_path}')

print(f'\nPer-method stats:')
for method in results_df['method'].unique():
    subset = results_df[results_df['method'] == method]
    print(f'  {method}: {len(subset)} frames, '
          f'avg improvement {subset["improvement"].mean():.2f}x, '
          f'avg sharpness {subset["orig_sharpness"].mean():.0f} → {subset["recon_sharpness"].mean():.0f}')

## 8. Visual Comparison

Side-by-side comparison of original vs reconstructed frames.
Focus on **logo visibility** — can you read the sponsor logos?

In [ ]:
n_preview = min(8, len(results_df))
preview_df = results_df.sort_values('improvement', ascending=False).head(n_preview)

cap = cv2.VideoCapture(str(VIDEO_PATH))

for _, row in preview_df.iterrows():
    fn = row['frame_num']
    fname = row['filename']

    cap.set(cv2.CAP_PROP_POS_FRAMES, fn)
    ret, original = cap.read()
    if not ret:
        continue

    recon_path = RECON_OUTPUT_DIR / fname
    if not recon_path.exists():
        continue
    reconstructed = cv2.imread(str(recon_path))

    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'ORIGINAL — sharpness={row["orig_sharpness"]:.0f}', fontsize=12)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(reconstructed, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'RECONSTRUCTED ({row["method"]}) — sharpness={row["recon_sharpness"]:.0f} ({row["improvement"]:.1f}x)', fontsize=12)
    axes[1].axis('off')
    plt.suptitle(f'Frame {fn} | {fname}', fontsize=10)
    plt.tight_layout()
    plt.show()

    # Center crop detail
    h, w = original.shape[:2]
    cy, cx = h // 2, w // 2
    ch, cw = h // 3, w // 3
    y1, y2 = cy - ch // 2, cy + ch // 2
    x1, x2 = cx - cw // 2, cx + cw // 2

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(cv2.cvtColor(original[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    axes[0].set_title('ORIGINAL (center crop)', fontsize=12)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(reconstructed[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    axes[1].set_title('RECONSTRUCTED (center crop)', fontsize=12)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print('---')

cap.release()

## 9. Sharpness Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(results_df['improvement'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(x=1.0, color='red', linestyle='--', label='No improvement')
axes[0].set_title('Sharpness Improvement (x times)')
axes[0].set_xlabel('Improvement factor')
axes[0].legend()

colors = ['green' if m == 'rvrt' else 'orange' for m in results_df['method']]
axes[1].scatter(results_df['orig_sharpness'], results_df['recon_sharpness'],
                c=colors, alpha=0.6)
max_val = max(results_df['orig_sharpness'].max(), results_df['recon_sharpness'].max())
axes[1].plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='No change')
axes[1].set_xlabel('Original Sharpness')
axes[1].set_ylabel('Reconstructed Sharpness')
axes[1].set_title('Original vs Reconstructed')
axes[1].legend(['No change', 'RVRT', 'NAFNet'])

method_counts = results_df['method'].value_counts()
axes[2].bar(method_counts.index, method_counts.values, color=['green', 'orange', 'gray'][:len(method_counts)])
axes[2].set_title('Frames per Method')
axes[2].set_ylabel('Count')

plt.suptitle(f'{MATCH_ID} — Reconstruction Results', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Upload to Roboflow (Optional)

Upload **reconstructed** frames instead of originals for annotation.

In [ ]:
import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass.getpass('Roboflow API Key: ')
ROBOFLOW_PROJECT = 'kit-sponsor-logos'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace().project(ROBOFLOW_PROJECT)

for fp in tqdm(sorted(RECON_OUTPUT_DIR.glob(f'{MATCH_ID}_*.jpg')), desc='Uploading'):
    try:
        project.upload(
            image_path=str(fp),
            split='train',
            tag_names=[f'{MATCH_ID},reconstructed,rvrt']
        )
    except Exception as e:
        print(f'Error: {e}')
        break

print('Upload complete.')